Simple notebook to load txt documents containing BBC news archive (http://mlg.ucd.ie/datasets/bbc.html).
- Leverages LLamaIndex SimpleDirectoryReader to handle recursive directories and file types
- Uses the Mistral Embeddings model to create vector representations of the text
- Stores text 'chunks' (nodes in LlamaIndex) along vectors into local FAISS storage (CPU)
- Smoke tests a few queries to test retrieval

In [1]:
from llama_index.llms.mistralai import MistralAI
from llama_index.embeddings.mistralai import MistralAIEmbedding
from llama_index.core.settings import Settings

In [2]:
import faiss

# 1024 for Mistral Embeddings
d = 1024

faiss_index = faiss.IndexFlatL2(d)

In [3]:
api_key = ""
llm = MistralAI(api_key=api_key,model="mistral-large-latest")
# batch size to 6 for mistral limits
embed_model = MistralAIEmbedding(model_name='mistral-embed', api_key=api_key, embed_batch_size=6)

Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 2048
Settings.chunk_overlap =  128

In [4]:
from llama_index.core import (
    SimpleDirectoryReader,
    load_index_from_storage,
    VectorStoreIndex,
    StorageContext,
)
from llama_index.vector_stores.faiss import FaissVectorStore

In [6]:
# load documents
documents = SimpleDirectoryReader("./data/bbc/", recursive=True).load_data(show_progress=True)

Loading files: 100%|██████████| 2225/2225 [00:00<00:00, 4881.15file/s]


In [7]:
vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context
)

In [8]:
# save index to disk
index.storage_context.persist()

In [9]:
# load index from disk
vector_store = FaissVectorStore.from_persist_dir("./storage")
storage_context = StorageContext.from_defaults(
    vector_store=vector_store, persist_dir="./storage"
)
index = load_index_from_storage(storage_context=storage_context)

In [10]:
query_engine = index.as_query_engine()
response = query_engine.query("What are Parmalat's profits?")

In [11]:
print(response)

Parmalat reported a profit of 77 million euros ($100 million) in the fourth quarter, which is double the profit from the same period in 2003.


In [12]:
query_engine = index.as_query_engine()
response = query_engine.query(
    "What can you tell me about Sarah Claxton?"
)

In [13]:
print(response)

Sarah Claxton is a British hurdler who, at 25 years old, is aiming for her first major medal at the European Indoor Championships in Madrid. She has set a new British record over 60m hurdles twice in the same season, with her best time being 7.96 seconds. Claxton has won the national 60m hurdles title for the past three years and is currently the equal fifth-fastest in the world for the event. She has recently focused solely on the hurdles, having previously also competed in the long jump.


In [21]:
for i in response.metadata.items():
    print(i[1]['file_path'])

/Users/paramsingh/Desktop/mistral/data/bbc/sport/001.txt
/Users/paramsingh/Desktop/mistral/data/bbc/sport/056.txt
